### Import relevant packages

In [1]:
import pandas as pd
import numpy as np 
from datetime import timedelta 
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

### Import datasets 

In [2]:
train_df = pd.read_csv('data/ais_train.csv', sep = "|")
vessels_df = pd.read_csv('data/vessels.csv', sep = '|')
ports_df = pd.read_csv('data/ports.csv', sep = '|')
ports_df['portLatitude'] = ports_df['latitude']
ports_df['portLongitude'] = ports_df['longitude']
ports_df = ports_df.drop(columns = ['latitude', 'longitude'])
test_df = pd.read_csv('data/ais_test.csv', sep = ',')
schedules_df = pd.read_csv('data/schedules_to_may_2024.csv', sep = '|')

### Some preprocessing 

In [3]:
train_df = pd.merge(train_df, ports_df, on = 'portId', how = 'left')

train_df = pd.merge(train_df, vessels_df[['vesselId', 'shippingLineId', ]], on = 'vesselId')


train_df['time'] = pd.to_datetime(train_df['time'])
test_df['time'] = pd.to_datetime(test_df['time'])

timestamps = train_df[['time', 'vesselId']]

#train_df['second'] = train_df['time'].dt.second
#train_df['minute'] = train_df['time'].dt.minute
#train_df['hour'] = train_df['time'].dt.hour
#train_df['day'] = train_df['time'].dt.day
#train_df['day_of_week'] = train_df['time'].dt.dayofweek
#train_df['month'] = train_df['time'].dt.month

#test_df['second'] = test_df['time'].dt.second
#test_df['minute'] = test_df['time'].dt.minute
#test_df['day'] = test_df['time'].dt.day

#test_df['day_of_week'] = test_df['time'].dt.dayofweek
#test_df['hour'] = test_df['time'].dt.hour
#test_df['month'] = test_df['time'].dt.month

train_df = train_df.drop(columns = ['countryName', 'ISO', 'UN_LOCODE', 'name', 'portLocation', ])

### Label encoding

In [4]:

le_vesselid = LabelEncoder()
all_vesselId = pd.concat([train_df['vesselId'], schedules_df['vesselId']], axis = 0)
le_vesselid.fit(all_vesselId)
train_df['vesselId'] = le_vesselid.transform(train_df['vesselId'])
test_df['vesselId'] =  le_vesselid.transform(test_df['vesselId'])
schedules_df['vesselId'] = le_vesselid.transform(schedules_df['vesselId'])

le_shippingLineId = LabelEncoder()
all_shippingLineId = pd.concat([train_df['shippingLineId'], schedules_df['shippingLineId']], axis = 0)
le_shippingLineId.fit(all_shippingLineId)
train_df['shippingLineId'] = le_shippingLineId.transform(train_df['shippingLineId'])
schedules_df['shippingLineId'] = le_shippingLineId.transform(schedules_df['shippingLineId'])

le_portid = LabelEncoder()
le_portid.fit(train_df['portId'])
train_df['portId'] = le_portid.transform(train_df['portId'])

train_df['navstat'] = pd.Categorical(train_df['navstat']).codes


### Feature engineering
First for the training set

In [5]:

X = train_df.sort_values(by = 'vesselId', kind =  'stable')
X_1 = X.copy()
X_1[['time_x', 'longitude', 'latitude']] = (X_1[['time', 'vesselId', 'longitude', 'latitude']].groupby(by = 'vesselId').shift(-1))
X_1['vesselId'] = X['vesselId']
X_1['time_diff'] =  (X_1['time_x'] - X_1['time']).dt.total_seconds()
X_1 = X_1.dropna()
display(X)
display(X_1)

,time,cog,sog,rot,heading,navstat,etaRaw,latitude,longitude,vesselId,portId,portLatitude,portLongitude,shippingLineId
131115,2024-01-12 14:07:47,308.1,17.1,-6,316,0,01-08 06:00,7.50361,77.58340,0,112,13.263333,80.341111,0
131279,2024-01-12 14:31:00,307.6,17.3,5,313,0,01-14 23:30,7.57302,77.49505,0,115,18.941944,72.885278,0
131514,2024-01-12 14:57:23,306.8,16.9,5,312,0,01-14 23:30,7.65043,77.39404,0,115,18.941944,72.885278,0
131696,2024-01-12 15:18:48,307.9,16.9,6,313,0,01-14 23:30,7.71275,77.31394,0,115,18.941944,72.885278,0
131885,2024-01-12 15:39:47,307.0,16.3,7,313,0,01-14 23:30,7.77191,77.23585,0,115,18.941944,72.885278,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1521244,2024-05-07 22:36:16,324.1,13.5,-2,325,0,05-08 03:00,59.63337,21.43237,699,78,60.437778,22.216389,5
1521409,2024-05-07 22:57:05,324.2,13.3,-3,326,0,05-08 03:00,59.69588,21.34225,699,78,60.437778,22.216389,5
1521625,2024-05-07 23:17:54,356.5,12.2,-1,354,0,05-08 03:00,59.76388,21.35317,699,78,60.437778,22.216389,5
1521821,2024-05-07 23:38:13,52.6,17.3,3,50,0,05-08 03:00,59.83316,21.38489,699,78,60.437778,22.216389,5


,time,cog,sog,rot,heading,navstat,etaRaw,latitude,longitude,vesselId,portId,portLatitude,portLongitude,shippingLineId,time_x,time_diff
131115,2024-01-12 14:07:47,308.1,17.1,-6,316,0,01-08 06:00,7.57302,77.49505,0,112,13.263333,80.341111,0,2024-01-12 14:31:00,1393.0
131279,2024-01-12 14:31:00,307.6,17.3,5,313,0,01-14 23:30,7.65043,77.39404,0,115,18.941944,72.885278,0,2024-01-12 14:57:23,1583.0
131514,2024-01-12 14:57:23,306.8,16.9,5,312,0,01-14 23:30,7.71275,77.31394,0,115,18.941944,72.885278,0,2024-01-12 15:18:48,1285.0
131696,2024-01-12 15:18:48,307.9,16.9,6,313,0,01-14 23:30,7.77191,77.23585,0,115,18.941944,72.885278,0,2024-01-12 15:39:47,1259.0
131885,2024-01-12 15:39:47,307.0,16.3,7,313,0,01-14 23:30,7.81285,77.18147,0,115,18.941944,72.885278,0,2024-01-12 15:54:48,901.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1521048,2024-05-07 22:15:17,296.3,14.7,3,298,0,05-08 03:00,59.63337,21.43237,699,78,60.437778,22.216389,5,2024-05-07 22:36:16,1259.0
1521244,2024-05-07 22:36:16,324.1,13.5,-2,325,0,05-08 03:00,59.69588,21.34225,699,78,60.437778,22.216389,5,2024-05-07 22:57:05,1249.0
1521409,2024-05-07 22:57:05,324.2,13.3,-3,326,0,05-08 03:00,59.76388,21.35317,699,78,60.437778,22.216389,5,2024-05-07 23:17:54,1249.0
1521625,2024-05-07 23:17:54,356.5,12.2,-1,354,0,05-08 03:00,59.83316,21.38489,699,78,60.437778,22.216389,5,2024-05-07 23:38:13,1219.0


In [ ]:
X = train_df.copy()
X[['time_y', 'longitude_y', 'latitude_y']] = (X[['time', 'vesselId', 'longitude', 'latitude']].groupby(by = 'vesselId').shift(-1))
X['vesselId'] = X['vesselId']
X['time_diff'] =  (X['time_y'] - X['time']).dt.total_seconds()
X = X.dropna()
for k in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:
    print(k , end = '\r')
    X_k = train_df.copy()
    X_k[['time_y', 'longitude_y', 'latitude_y']] = (X_k[['time', 'vesselId', 'longitude', 'latitude']].groupby(by = 'vesselId').shift(-2**k))
    X_k['vesselId'] = train_df['vesselId']
    X_k['time_diff'] =  (X_k['time_y'] - X_k['time']).dt.total_seconds()
    X_k = X_k.dropna()
    X = pd.concat([X, X_k])

display(X)

### Feature engineering 2 
For test set 

In [ ]:
display(test_df)

vessels = test_df['vesselId'].unique()
last_values = {}
for vessel in vessels:
    data_vessel = train_df[train_df['vesselId'] == vessel]
    last_values[vessel] = data_vessel.tail(1)


In [ ]:
test_data = {}
for index, row in test_df.iterrows():
    print(index, end = '\r')
    last_data = last_values[row['vesselId']]
    row = pd.DataFrame(row).T    
    row = pd.merge(last_data, row, left_on='vesselId', right_on='vesselId', how='right')
    test_data[index] = row
test_data = pd.concat(test_data)

In [ ]:
test_data['time_diff'] = (pd.to_datetime(test_data['time_y']) - pd.to_datetime(test_data['time_x'])).dt.total_seconds()
display(test_data)

In [ ]:
X['time_x'] = X['time']
features1 = ['time_x', 'time_y', 'time_diff', 'cog', 'sog', 'rot', 'heading', 'navstat', 'etaRaw', 'latitude', 'longitude', 'portLatitude', 'portLongitude']
features2 = ['time_x', 'time_y', 'time_diff', 'cog', 'sog', 'rot', 'heading', 'navstat', 'etaRaw', 'latitude', 'longitude', 'portLatitude', 'portLongitude', 'latitude_y', 'longitude_y']

display(test_data[features1])
display(X[features2])


In [ ]:
train_data = X.copy()
features = ['time_diff', 'vesselId', 'cog', 'sog', 'rot', 'heading', 'latitude', 'longitude', 'portLatitude', 'portLongitude']
targets = ['latitude_y', 'longitude_y']

x = train_data[features]
y = train_data[targets]
display(x)
display(y)
params = {
    'objective': 'reg:squarederror',
    'max_depth': 13,
    'eta':  0.01,
    'n_jobs': -1
}

dtrain = xgb.DMatrix(x, label = y)
evals = [(dtrain, 'train')]
model = xgb.train(params, dtrain, num_boost_round= 400, evals = evals, verbose_eval= 10)


In [ ]:
test_data[features]

In [ ]:
test_data['vesselId'] = test_data['vesselId'].apply(lambda x: int(x))
dtest = xgb.DMatrix(test_data[features])

prediction = model.predict(dtest)

In [ ]:
test_df['latitude_predicted'] = prediction[:,0]
test_df['longitude_predicted'] = prediction[:,1]

display(test_df)

submission = pd.DataFrame(test_df[['ID','longitude_predicted', 'latitude_predicted']])
submission.to_csv('submission_4_11.csv', index = False)

In [ ]:
vessel = test_df['vesselId'].unique()[2]
vessel_data_test = test_df[test_df['vesselId'] == vessel]
vessel_data_train = train_df[train_df['vesselId'] == vessel]
plt.plot(vessel_data_test['time'],vessel_data_test['longitude_predicted'])
plt.plot(vessel_data_train['time'],vessel_data_train['longitude'])

plt.plot(vessel_data_test['time'],vessel_data_test['latitude_predicted'])
plt.plot(vessel_data_train['time'],vessel_data_train['latitude'])